In [0]:
%pip install -U -qqqq unitycatalog-ai[databricks] mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv
%restart_python

In [0]:
import sys
sys.path.append(".")

from config import *

In [0]:
%sql
use identifier(:WORKSHOP_CATALOG || "." || :WORKSHOP_SCHEMA);

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_shipments_in_transit(
  dest_city STRING COMMENT 'Destination city or region name to filter active shipments (e.g., New York, NY, or NYC). The function will match common synonyms and partial names.'
)
RETURNS TABLE(
  shipment_id STRING COMMENT 'Unique shipment identifier.',
  product_id STRING COMMENT 'Unique product identifier for the shipped item.',
  product_name STRING COMMENT 'Descriptive name of the product.',
  supplier_id STRING COMMENT 'Unique identifier for the supplier sending the shipment.',
  destination STRING COMMENT 'Full destination city or location for the shipment.',
  eta_date DATE COMMENT 'Estimated delivery date for the shipment.',
  status STRING COMMENT 'Current shipment status (e.g., in-transit, delivered, delayed).',
  temperature_max_f DOUBLE COMMENT 'Maximum safe transport temperature threshold in Fahrenheit.'
)
LANGUAGE SQL
COMMENT 'Returns all shipments currently in transit to a given destination city. Matches common aliases such as New York, NY, and NYC.'
RETURN
(
  SELECT
    shipment_id,
    product_id,
    product_name,
    supplier_id,
    destination,
    CAST(eta_date AS DATE) AS eta_date,
    status,
    temperature_max_f
  FROM shipments
  WHERE LOWER(status) = 'in-transit'
    AND (
      LOWER(destination) LIKE CONCAT('%', LOWER(dest_city), '%')
      OR (
        LOWER(dest_city) IN ('ny', 'nyc', 'new york', 'new york city')
        AND LOWER(destination) LIKE '%new york%'
      )
    )
  ORDER BY eta_date ASC
);

In [0]:
%sql
SELECT * FROM get_shipments_in_transit('New York');

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_supplier_details(
  supplier_name STRING COMMENT 'Full or partial name of the supplier to look up. The function performs a case-insensitive match (e.g., "Zimmer", "zimmer biotech", or "biotech").'
)
RETURNS TABLE(
  supplier_id STRING COMMENT 'Unique identifier of the supplier in the master supplier table.',
  supplier_name STRING COMMENT 'Registered name of the supplier organization.',
  contact_name STRING COMMENT 'Primary supplier contact person for escalations or logistics issues.',
  contact_email STRING COMMENT 'Email address of the supplier contact to be used for escalations or updates.',
  phone STRING COMMENT 'Direct phone number for the supplier contact or escalation hotline.',
  tier STRING COMMENT 'Escalation tier or relationship classification (e.g., Tier 1 = strategic supplier, Tier 2 = secondary supplier).'
)
LANGUAGE SQL
COMMENT 'Retrieves supplier contact and escalation details based on a given supplier name. Performs a case-insensitive partial match for flexibility in search.'
RETURN
(
  SELECT
    supplier_id,
    supplier_name,
    contact_name,
    contact_email,
    phone,
    tier
  FROM suppliers
  WHERE LOWER(supplier_name) LIKE LOWER(CONCAT('%', supplier_name, '%'))
  LIMIT 1
);

In [0]:
%sql
SELECT * FROM get_supplier_details('Zimmer Biotech');

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_backup_inventory(
  product_id STRING COMMENT 'Unique product identifier for which to locate alternate or backup inventory. Accepts full or partial product IDs.'
)
RETURNS TABLE(
  site_id STRING COMMENT 'Unique identifier of the warehouse or distribution site holding the product.',
  site_name STRING COMMENT 'Descriptive name of the warehouse or storage facility.',
  city STRING COMMENT 'City where the warehouse is located.',
  state STRING COMMENT 'State or region of the warehouse location.',
  product_id STRING COMMENT 'Unique identifier of the product in inventory.',
  product_name STRING COMMENT 'Official product name or description.',
  supplier_id STRING COMMENT 'Unique identifier of the supplier providing this product.',
  on_hand_qty INT COMMENT 'Current quantity of the product available for immediate dispatch.',
  lot_expiry DATE COMMENT 'Expiration date of the product lot, used to prioritize shipments by freshness.'
)
LANGUAGE SQL
COMMENT 'Returns all available backup inventory records for a given product across warehouses, filtered by product ID and ordered by earliest lot expiry date.'
RETURN
(
  SELECT
    site_id,
    site_name,
    city,
    state,
    product_id,
    product_name,
    supplier_id,
    on_hand_qty,
    CAST(lot_expiry AS DATE) AS lot_expiry
  FROM inventory
  WHERE LOWER(inventory.product_id) LIKE LOWER(CONCAT('%', product_id, '%'))
    AND on_hand_qty > 0
  ORDER BY lot_expiry ASC
);

In [0]:
%sql
SELECT * FROM get_backup_inventory('PRD-001');

In [0]:
%sql
CREATE OR REPLACE FUNCTION temp_gap(
  weather_f DOUBLE COMMENT 'Current or forecasted ambient temperature in degrees Fahrenheit at the shipment destination.',
  max_temp_f DOUBLE COMMENT 'Maximum allowable temperature in degrees Fahrenheit for the product to remain within safe limits.'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Calculates the absolute difference (in °F) between current weather and a product’s maximum safe temperature threshold. Positive values indicate risk of overexposure or cold-chain failure.'
RETURN ABS(max_temp_f - weather_f);

In [0]:
%sql
SELECT
  shipment_id,
  temperature_max_f,
  temp_gap(50, temperature_max_f) AS diff
FROM shipments;